# 03 · Las diferencias que sobreviven

**Objetivo:** Medir cuánto de una diferencia de precio entre municipios se mantiene cuando se comparan productos, presentaciones y periodos equivalentes, y distinguir lo que los datos públicos permiten afirmar de lo que haría falta para evaluar una oportunidad de negocio.

**Pregunta:** ¿Qué diferencias sobreviven, en qué productos y municipios, y con qué incertidumbre?

Por qué importa: una diferencia que sobrevive a la validación y a la prueba contra el azar es la única que
vale la pena investigar con costos, márgenes y demanda.

**Decisión:** para cada artículo comparable (mismo nombre, presentación, marca y giro, con 3 tiendas o más en 10
municipios o más) se prueba si sus precios de tienda difieren entre municipios más que entre tiendas del mismo
municipio (Kruskal-Wallis), con control de falsos positivos de Benjamini-Hochberg al 10%. Alternativa descartada: la
prueba sin corrección, que con más de mil artículos declara «distintos» a unos 50 solo por azar.

In [1]:
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(_v, "1")
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import entorno
entorno.setup()
import profeco, tablas, estadistica as E, charts as C
import embudo as M
PROC, META, CHARTS, SOCIAL = entorno.PROC, entorno.META, entorno.CHARTS, entorno.SOCIAL
CFG = entorno.config()
def reg(*a, **k):
    entorno.registrar(*a, **k)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)
raw = entorno.load_raw()
d_all = M.prep(raw)
d = d_all[~d_all.catalogo.isin(M.FUERA_PRINCIPAL)].copy()      # universo principal
params = json.loads((META / "parametros.json").read_text(encoding="utf-8"))
REPS = params.get("bootstrap_replicas", 500)
C.register_cities(d_all.municipio_id.unique())
res, prod, un, pm = M.embudo(d)
U = set(prod.index)
fin = un["extremos"][un["extremos"].producto.isin(U)]
sp = M.tiendas_finales(d).merge(fin[M.UNIDAD], on=M.UNIDAD)
kw = M.prueba_municipio(sp)
kw["sobrevive"] = M.bh(kw.p_valor, params["fdr"])
ic = M.boot_unidad(pm["extremos"].merge(fin[M.UNIDAD], on=M.UNIDAD), reps=REPS)
mc = M.misma_cadena(sp).rename(columns={"brecha_pct": "misma_cadena_pct"})[M.UNIDAD + ["misma_cadena_pct"]]
az = pd.read_csv(PROC / "azar_unidades.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
for c in ["brecha_pct", "azar_mediana", "azar_p95", "p_azar", "exceso_pp"]:
    az[c] = az[c].astype(float)
f = (fin.merge(kw, on=M.UNIDAD).merge(ic, on=M.UNIDAD).merge(mc, on=M.UNIDAD, how="left")
        .merge(az[M.UNIDAD + ["azar_mediana", "azar_p95"]], on=M.UNIDAD, how="left"))
info = d.groupby(M.UNIDAD).agg(catalogo=("catalogo", "first"), categoria=("categoria", "first")).reset_index()
f = f.merge(info, on=M.UNIDAD, how="left")
F = "03_diferencias_que_sobreviven"
print(f"Artículos comparables: {len(f):,} · p<0.05 sin corrección: {(f.p_valor < .05).sum()} · sobreviven con corrección: {f.sobrevive.sum()}")
reg("n_articulos", len(f), fuente=F)
reg("n_sobreviven", f.sobrevive.sum(), n=len(f), fuente=F, nota="artículos con diferencia entre municipios tras Benjamini-Hochberg 10%")
reg("pct_sobreviven", f.sobrevive.mean() * 100, n=len(f), fuente=F)
reg("n_p05", (f.p_valor < .05).sum(), n=len(f), fuente=F)
reg("esperados_azar_p05", len(f) * 0.05, fuente=F)

Artículos comparables: 1,238 · p<0.05 sin corrección: 138 · sobreviven con corrección: 45


## 1. ¿Qué tienen en común los que sobreviven?

**Familia** = agrupación para leer el resultado: bebidas (refresco, cerveza, agua, jugo), tortilla, frescos y a granel
(frutas, verduras, carne, pan de mostrador, huevo) y otros.

In [2]:
def familia(row):
    p = row.producto.lower()
    if any(k in p for k in ["refresco", "cerveza", "agua", "jugo"]):
        return "Bebidas"
    if "tortilla" in p:
        return "Tortilla"
    if row.catalogo in ("Frutas y Legumbres", "Pescados y Mariscos", "Mercados") or any(k in p for k in ["carne", "bolillo", "huevo", "pollo"]):
        return "Frescos y a granel"
    return "Otros"
f["familia"] = f.apply(familia, axis=1)
fam = f.groupby("familia").agg(articulos=("sobrevive", "size"), sobreviven=("sobrevive", "sum"))
fam["% que sobrevive"] = fam.sobreviven / fam.articulos * 100
display(fam.round(1))
cmp_ = f.groupby("sobrevive")[["brecha_pct", "azar_mediana", "misma_cadena_pct", "municipios"]].median()
display(cmp_.round(1))
for k in fam.index:
    key = {"Bebidas": "bebidas", "Tortilla": "tortilla", "Frescos y a granel": "frescos", "Otros": "otros"}[k]
    reg(f"sob_{key}", fam.loc[k, "sobreviven"], n=fam.loc[k, "articulos"], fuente=F)
    reg(f"pct_sob_{key}", fam.loc[k, "% que sobrevive"], n=fam.loc[k, "articulos"], fuente=F)
reg("brecha_sobreviven", cmp_.loc[True, "brecha_pct"], fuente=F)
reg("brecha_no_sobreviven", cmp_.loc[False, "brecha_pct"], fuente=F)
reg("cadena_sobreviven", cmp_.loc[True, "misma_cadena_pct"], fuente=F, nota="brecha entre municipios de la misma cadena, artículos que sobreviven")
reg("cadena_no_sobreviven", cmp_.loc[False, "misma_cadena_pct"], fuente=F)

,articulos,sobreviven,% que sobrevive
familia,,,
Bebidas,80,22,27.5
Frescos y a granel,138,12,8.7
Otros,1015,9,0.9
Tortilla,5,2,40.0


,brecha_pct,azar_mediana,misma_cadena_pct,municipios
sobrevive,,,,
False,10.0,9.4,1.3,29.0
True,23.2,10.4,13.3,40.0


**Lectura.** En los artículos que sobreviven, la misma cadena también cobra distinto según el municipio: la
diferencia no es solo qué cadenas hay en cada lugar. En los que no sobreviven, la misma cadena cobra casi igual en
todas partes.

## 2. Dónde está la diferencia: municipio caro y barato típicos

In [3]:
pmf = pm["extremos"].merge(f.loc[f.sobrevive, M.UNIDAD], on=M.UNIDAD)
def extremos_mun(g):
    g = g.sort_values("precio")
    q10, q90 = g.precio.quantile(.1), g.precio.quantile(.9)
    lo = g.iloc[(g.precio - q10).abs().argmin()]
    hi = g.iloc[(g.precio - q90).abs().argmin()]
    return pd.Series({"mun_barato": C.city(lo.municipio_id), "precio_barato": lo.precio, "mun_caro": C.city(hi.municipio_id),
                      "precio_caro": hi.precio, "mediana": g.precio.median()})
ext = pmf.groupby(M.UNIDAD).apply(extremos_mun, include_groups=False).reset_index()
sob = f[f.sobrevive].merge(ext, on=M.UNIDAD)
# Relevante: además de pasar la prueba, la diferencia es de 10% o más y su IC90 queda arriba de lo típico del azar.
sob["relevante"] = (sob.brecha_pct >= M.MIN_RELEVANTE) & (sob.ic90_inf > sob.azar_mediana)
sob["etiqueta"] = [M.etiqueta(a, b_, c_, e_) for a, b_, c_, e_ in zip(sob.producto, sob.pres_key, sob.marca, sob.presentacion)]
sob = sob.sort_values(["relevante", "p_valor"], ascending=[False, True])
cols = ["etiqueta", "producto", "pres_key", "presentacion", "marca", "giro", "familia", "municipios", "brecha_pct", "ic90_inf", "ic90_sup",
        "azar_mediana", "azar_p95", "misma_cadena_pct", "mun_barato", "precio_barato", "mun_caro", "precio_caro", "p_valor", "relevante"]
display(sob[cols].round(2))
rel = sob[sob.relevante]
reg("n_relevantes", len(rel), n=len(sob), fuente=F, nota="sobreviven, brecha >= 10% e IC90 inferior arriba de la mediana del azar")
for fam_, key in [("Bebidas", "bebidas"), ("Tortilla", "tortilla"), ("Frescos y a granel", "frescos"), ("Otros", "otros")]:
    reg(f"rel_{key}", (rel.familia == fam_).sum(), fuente=F)
reg("brecha_relevantes", rel.brecha_pct.median(), n=len(rel), fuente=F)
reg("cadena_relevantes", rel.misma_cadena_pct.median(), n=len(rel), fuente=F)
f.to_csv(PROC / "articulos_prueba.csv", index=False, encoding="utf-8-sig")
sob[cols].to_csv(PROC / "sobrevivientes.csv", index=False, encoding="utf-8-sig")

,etiqueta,producto,pres_key,presentacion,marca,giro,familia,municipios,brecha_pct,ic90_inf,ic90_sup,azar_mediana,azar_p95,misma_cadena_pct,mun_barato,precio_barato,mun_caro,precio_caro,p_valor,relevante
43,"Tortilla de Maíz (1 kg, granel)",Tortilla de Maíz,1.0 kg,1 Kg. Granel,S/M,Tortillerías,Tortilla,57,46.83,36.15,50.47,18.09,21.73,46.86,Iztapalapa,20.50,Benito Juárez (Quintana Roo),29.96,0.0,True
30,Refresco Coca Cola Zero (botella 600 ml),Refresco,0.6 l,Botella 600 Ml.,Coca Cola. Zero,Supermercado / Tienda de Autoservicio,Bebidas,45,27.26,20.16,35.49,19.20,24.21,18.95,Miguel Hidalgo,16.49,San Luis Potosí,20.99,0.0,True
37,"Refresco Fresca (botella 3 lt, no retornable)",Refresco,3.0 l,Botella 3 Lt. No Retornable,Fresca,Supermercado / Tienda de Autoservicio,Bebidas,41,26.40,16.57,31.82,9.54,15.00,22.79,Benito Juárez (Ciudad de México),41.96,Culiacán,53.04,0.0,True
39,"Refresco Sprite (botella 3 lt, no retornable)",Refresco,3.0 l,Botella 3 Lt. No Retornable,Sprite,Supermercado / Tienda de Autoservicio,Bebidas,38,23.24,7.64,29.97,5.51,9.95,22.15,Benito Juárez (Ciudad de México),41.97,Hermosillo,51.17,0.0,True
16,Cerveza Tecate (paquete con 12 latas de 355 ml),Cerveza,0.355 l,Paquete con 12 Latas de 355 Ml.,Tecate,Supermercado / Tienda de Autoservicio,Bebidas,43,26.10,14.79,33.54,14.70,17.91,16.80,La Paz,172.88,Tlalnepantla,219.01,0.0,True
36,"Refresco Jarritos (botella 2 lt, no retornable, tamarindo)",Refresco,2.0 l,Botella 2 Lt. No Retornable. Tamarindo,Jarritos,Supermercado / Tienda de Autoservicio,Bebidas,40,20.30,15.75,21.12,13.79,17.38,16.10,Venustiano Carranza,24.90,Guadalajara,29.96,0.0,True
14,"Cerveza Bohemia Pilsner (canasta con 6 botellas de 355 ml, no reto...",Cerveza,0.355 l,Canasta con 6 Botellas de 355 Ml. No Retornable (Clara),Bohemia. Pilsner,Supermercado / Tienda de Autoservicio,Bebidas,41,16.38,16.22,19.43,12.45,16.23,16.21,Naucalpan de Juárez,128.92,Tampico,150.04,0.0,True
15,Cerveza Indio (paquete con 12 latas de 355 ml),Cerveza,0.355 l,Paquete con 12 Latas de 355 Ml.,Indio,Supermercado / Tienda de Autoservicio,Bebidas,32,34.50,24.84,41.81,22.28,30.90,25.74,Hermosillo,163.02,Iztapalapa,219.43,0.0,True
32,"Refresco Fresca (botella 2 lt, no retornable)",Refresco,2.0 l,Botella 2 Lt. No Retornable,Fresca,Supermercado / Tienda de Autoservicio,Bebidas,33,11.40,9.73,33.37,9.35,12.40,9.84,Tlalnepantla,30.47,León,34.07,0.0,True
4,"Alimentos Preparados Maruchan (vaso 64 gr, sopa instantánea con ca...",Alimentos Preparados,0.064 kg,Vaso 64 Gr. Sopa Instantánea con Camarón y Chile Piquín,Maruchan,Supermercado / Tienda de Autoservicio,Otros,46,26.65,23.44,35.12,22.78,26.51,24.93,Miguel Hidalgo,14.99,Tampico,18.99,0.0,True


## 3. Casos que sobreviven

**Decisión:** un caso por familia (bebidas, tortilla, frescos), el relevante de menor valor p en cada una. Se reporta
la brecha con su IC90 y la referencia de azar del mismo artículo.

In [4]:
casos = []
for fam_, key in [("Tortilla", "tortilla"), ("Bebidas", "bebida"), ("Frescos y a granel", "fresco")]:
    s = sob[(sob.familia == fam_) & sob.relevante]
    if not len(s):
        continue
    c = s.iloc[0]
    casos.append(c)
    giro_txt = M.giro_corto(c.giro, plural=True)
    reg(f"caso_{key}_brecha", c.brecha_pct, (c.ic90_inf, c.ic90_sup), n=c.municipios, fuente=F,
        nota=f"{c.producto} · {c.pres_key} · {c.marca} · {giro_txt}")
    reg(f"caso_{key}_nombre", f"{c.etiqueta} en {giro_txt}", fuente=F)
    reg(f"caso_{key}_mun_barato", c.mun_barato, fuente=F)
    reg(f"caso_{key}_mun_caro", c.mun_caro, fuente=F)
    reg(f"caso_{key}_municipios", c.municipios, fuente=F)
    reg(f"caso_{key}_azar", c.azar_mediana, fuente=F)
    reg(f"caso_{key}_cadena", c.misma_cadena_pct, fuente=F)
    reg(f"caso_{key}_barato", c.precio_barato, fuente=F, nota=c.mun_barato)
    reg(f"caso_{key}_caro", c.precio_caro, fuente=F, nota=c.mun_caro)
casos = pd.DataFrame(casos)
display(casos[cols].round(2))
casos[cols].to_csv(PROC / "casos_sobreviven.csv", index=False, encoding="utf-8-sig")

,etiqueta,producto,pres_key,presentacion,marca,giro,familia,municipios,brecha_pct,ic90_inf,ic90_sup,azar_mediana,azar_p95,misma_cadena_pct,mun_barato,precio_barato,mun_caro,precio_caro,p_valor,relevante
43,"Tortilla de Maíz (1 kg, granel)",Tortilla de Maíz,1.0 kg,1 Kg. Granel,S/M,Tortillerías,Tortilla,57,46.83,36.15,50.47,18.09,21.73,46.86,Iztapalapa,20.50,Benito Juárez (Quintana Roo),29.96,0.0,True
30,Refresco Coca Cola Zero (botella 600 ml),Refresco,0.6 l,Botella 600 Ml.,Coca Cola. Zero,Supermercado / Tienda de Autoservicio,Bebidas,45,27.26,20.16,35.49,19.20,24.21,18.95,Miguel Hidalgo,16.49,San Luis Potosí,20.99,0.0,True
23,"Lechuga (pieza, romana)",Lechuga,texto: Pieza. Romana,Pieza. Romana,S/M,Supermercado / Tienda de Autoservicio,Frescos y a granel,45,38.80,36.15,47.37,33.89,37.87,25.54,Culiacán,17.90,Benito Juárez (Ciudad de México),24.87,0.0,True


## 4. Lo que no resultó sólido

- **Distintos pero pequeños:** pasan la prueba, pero la diferencia es menor a 10% o su IC90 toca lo típico del azar.
  Estadísticamente distintos no es lo mismo que relevantes para el negocio.
- **Distintos sin corregir:** con la prueba sin corrección (valor p < 0.05) parecerían distintos, pero no sobreviven al
  control de falsos positivos.

In [5]:
fragil = sob[~sob.relevante]
reg("sob_no_relevantes", len(fragil), n=len(sob), fuente=F, nota="sobreviven a la prueba pero con brecha < 10% o IC90 que toca la mediana del azar")
reg("p05_no_sobreviven", ((f.p_valor < .05) & ~f.sobrevive).sum(), fuente=F)
display(fragil[cols].round(2))
v = entorno.cifras()
display(Markdown(f'''**Conclusión.** De {v["n_articulos"]:,.0f} artículos comparables, **{v["n_sobreviven"]:.0f}
({v["pct_sobreviven"]:.0f}%)** tienen una diferencia entre municipios que supera la prueba con control de falsos
positivos, y solo **{v["n_relevantes"]:.0f}** con una diferencia de 10% o más claramente arriba del azar:
{v["rel_bebidas"]:.0f} bebidas, {v["rel_frescos"]:.0f} frescos, {v["rel_tortilla"]:.0f} de tortilla y {v["rel_otros"]:.0f} de otros
productos. En los que sobreviven, la misma cadena cobra {v["cadena_sobreviven"]:.0f}% más en el municipio caro típico,
contra {v["cadena_no_sobreviven"]:.1f}% en los demás. {v["p05_no_sobreviven"]:.0f} artículos parecían distintos sin corregir y
no sobreviven.'''))

,etiqueta,producto,pres_key,presentacion,marca,giro,familia,municipios,brecha_pct,ic90_inf,ic90_sup,azar_mediana,azar_p95,misma_cadena_pct,mun_barato,precio_barato,mun_caro,precio_caro,p_valor,relevante
41,"Refresco Fanta (botella 3 lt, no retornable, naranja)",Refresco,3.0 l,Botella 3 Lt. No Retornable. Naranja,Fanta,Supermercado / Tienda de Autoservicio,Bebidas,34,27.13,5.85,29.98,8.62,13.88,18.43,Cuernavaca,41.95,La Paz,53.50,0.0,False
33,"Refresco Sprite (botella 2 lt, no retornable)",Refresco,2.0 l,Botella 2 Lt. No Retornable,Sprite,Supermercado / Tienda de Autoservicio,Bebidas,33,9.88,9.72,32.32,7.65,10.45,9.96,Huixquilucan,30.97,León,34.03,0.0,False
34,"Refresco Sidral Mundet (botella 2 lt, no retornable)",Refresco,2.0 l,Botella 2 Lt. No Retornable.,Sidral Mundet,Supermercado / Tienda de Autoservicio,Bebidas,33,9.90,9.73,29.14,9.47,10.45,11.52,Atizapán de Zaragoza,30.95,Oaxaca de Juárez,34.03,0.0,False
1,"Agua Sin Gas Bonafont (garrafón 20 lt, (sólo liquido))",Agua Sin Gas,20.0 l,Garrafón 20 Lt. (Sólo Liquido),Bonafont,Supermercado / Tienda de Autoservicio,Bebidas,19,10.06,5.99,12.77,7.27,9.56,6.01,León,59.00,Benito Juárez (Ciudad de México),64.93,0.0,False
29,"Plátano (1 kg, granel, macho)",Plátano,1.0 kg,1 Kg. Granel. Macho,S/M,Supermercado / Tienda de Autoservicio,Frescos y a granel,41,20.03,15.77,23.55,17.18,19.94,13.61,Campeche,34.91,Benito Juárez (Ciudad de México),41.90,0.0,False
35,"Refresco Fanta (botella 2 lt, no retornable, naranja)",Refresco,2.0 l,Botella 2 Lt. No Retornable. Naranja,Fanta,Supermercado / Tienda de Autoservicio,Bebidas,31,9.83,9.70,24.29,9.77,11.36,9.43,Benito Juárez (Ciudad de México),30.97,León,34.01,0.0,False
38,"Refresco Red Cola (botella 3 lt, no retornable)",Refresco,3.0 l,Botella 3 Lt. No Retornable,Red Cola,Supermercado / Tienda de Autoservicio,Bebidas,20,27.30,0.27,30.88,1.99,11.73,27.43,Benito Juárez (Ciudad de México),34.97,Zacatecas,44.46,0.0,False
5,"Alimentos Preparados Maruchan (vaso 64 gr, sopa instantánea, sabor...",Alimentos Preparados,0.064 kg,Vaso 64 Gr. Sopa Instantánea. Sabor a Pollo,Maruchan,Supermercado / Tienda de Autoservicio,Otros,43,26.54,18.89,27.64,19.73,25.64,19.27,Tlalnepantla,15.00,Tampico,18.99,0.0,False
22,"Jugo de Fruta Boing! (caja 1 lt, mango)",Jugo de Fruta,1.0 l,Caja 1 Lt. Mango,Boing!,Supermercado / Tienda de Autoservicio,Bebidas,43,11.84,4.92,15.95,5.75,11.47,2.19,Azcapotzalco,30.45,Hermosillo,34.07,0.0,False
21,Huevo Bachoco (paquete c/18 blanco),Huevo,18.0 pieza,Paquete C/18 Blanco,Bachoco,Supermercado / Tienda de Autoservicio,Frescos y a granel,44,15.00,12.31,27.62,13.14,15.04,5.31,Nezahualcóyotl,40.01,Solidaridad,46.03,0.0,False


**Conclusión.** De 1,238 artículos comparables, **45
(4%)** tienen una diferencia entre municipios que supera la prueba con control de falsos
positivos, y solo **21** con una diferencia de 10% o más claramente arriba del azar:
10 bebidas, 6 frescos, 2 de tortilla y 3 de otros
productos. En los que sobreviven, la misma cadena cobra 13% más en el municipio caro típico,
contra 1.3% en los demás. 93 artículos parecían distintos sin corregir y
no sobreviven.